<a href="https://colab.research.google.com/github/gpugovka/ML-stuff/blob/main/House_Sale_Price_Prediction_Erin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **HW3**

Use the final Ames Housing Dataset you have processed and saved and apply any regression + regularization model on that use Grid Search and Cross Validation to find the best parameters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


Loading the data with preprocessed features from the results of assignment Feature Engineering and Data Preparation_Exercises_Canvas_Erin.ipynb.

It already has the missing data handled and contains only numeric features.

In [ ]:
df = pd.read_csv("AMES_Final_DF.csv")
df.head()

,PID,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,...,Sale Type_ConLw,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,526301100,141.0,31770,6,5,1960,1960,112.0,639.0,0.0,...,False,False,False,False,True,False,False,False,True,False
1,526350040,80.0,11622,5,6,1961,1961,0.0,468.0,144.0,...,False,False,False,False,True,False,False,False,True,False
2,526351010,81.0,14267,6,6,1958,1958,108.0,923.0,0.0,...,False,False,False,False,True,False,False,False,True,False
3,526353030,93.0,11160,7,5,1968,1968,0.0,1065.0,0.0,...,False,False,False,False,True,False,False,False,True,False
4,527105010,74.0,13830,5,5,1997,1998,0.0,791.0,0.0,...,False,False,False,False,True,False,False,False,True,False


In [ ]:
df.describe()

,PID,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,...,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,SalePrice
count,2.925000e+03,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,...,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000,2925.000000
mean,7.141299e+08,69.683099,10109.861538,6.091624,5.563761,1971.323761,1984.229060,100.318632,438.974359,49.790427,...,93.487521,47.210598,22.978462,2.596923,16.029402,2.083077,44.909744,6.216752,2007.791111,180815.537436
std,1.887183e+08,21.344161,7785.509739,1.406386,1.112262,30.228826,20.857822,176.667803,439.426274,169.274143,...,126.146364,66.562829,64.093003,25.162589,56.131397,34.514231,472.237990,2.712550,1.317483,79951.696564
min,5.263011e+08,0.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,12789.000000
25%,5.284770e+08,60.000000,7438.000000,5.000000,5.000000,1954.000000,1965.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000,129500.000000
50%,5.354532e+08,70.000000,9430.000000,6.000000,5.000000,1973.000000,1993.000000,0.000000,370.000000,0.000000,...,0.000000,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,160000.000000
75%,9.071800e+08,80.000000,11526.000000,7.000000,6.000000,2000.000000,2004.000000,162.000000,734.000000,0.000000,...,168.000000,70.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,213500.000000
max,1.007100e+09,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,2288.000000,1526.000000,...,1424.000000,742.000000,1012.000000,508.000000,576.000000,800.000000,15500.000000,12.000000,2010.000000,755000.000000


Confirm there's no missing data

In [ ]:
def percent_missing(df):
  missing_percent = df.isnull().sum() / len(df) * 100
  return missing_percent[missing_percent > 0].sort_values(ascending=True)

missing_data_stats = percent_missing(df)
missing_data_stats

,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Columns: 264 entries, PID to Sale Condition_Partial
dtypes: bool(227), float64(11), int64(26)
memory usage: 1.5 MB


Let's select the features with strong negative and positive correlation to use by the model.

In [ ]:
cor_mat = df.corr()['SalePrice'].sort_values()
selected_features = cor_mat[abs(cor_mat) > 0.25]
selected_features

,SalePrice
Exter Qual_TA,-0.591459
Kitchen Qual_TA,-0.527461
Bsmt Qual_TA,-0.453022
Garage Finish_Unf,-0.422363
Garage Type_Detchd,-0.365209
Foundation_CBlock,-0.350087
Heating QC_TA,-0.338812
Lot Shape_Reg,-0.300738
MS Zoning_RM,-0.292661
Bsmt Exposure_No,-0.283246


Select a subset of well corellated features. Define X features and y labels

In [ ]:
df_filtered = df[selected_features.index]

X = df_filtered.drop('SalePrice', axis=1)
y = df_filtered['SalePrice']

In [ ]:
X.shape, y.shape

((2925, 52), (2925,))

Data Splitting: Data is divided into training and test sets to evaluate final generalization. The training set fits the model parameters, and the test set estimates performance on held-out data.  We won't use the test set in training to avoid data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2047, 52), (878, 52), (2047,), (878,))

Scale the features using the standard scaler from the library. I tried to train as is from there, but apparently the labels are huge comparing to the features due to they are in millions of dollars, which scaled the errors even though learning was fine. To address that I decided to take Log() from the labels and when I calculate the final labels I use Exp() to translate them back to the millions of dollars unit.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

**Cross-Validation**: The model uses 5-fold cross-validation to average validation errors across multiple partitions of the training set. This step reduces variance in the performance estimate.

**Regularization**: Regularization prevents overfitting by penalizing parameter magnitudes. L_1 regularization forces uninformative coefficients to exactly zero, while L_2 regularization smoothly shrinks weights.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

def find_and_fit_best_model(model, param_grid, X_train_, y_train_):
  number_of_folds = 5

  grid_search = GridSearchCV(estimator=model,
                             param_grid=param_grid,
                             scoring='neg_mean_squared_error',
                             cv=number_of_folds,
                             verbose=0)
  grid_search.fit(X_train_, y_train_)

  best_model = grid_search.best_estimator_
  best_params = grid_search.best_params_

  print("Best Model:", best_model)
  print("Best Parameters:", best_params)

  best_model.fit(X_train_, y_train_)
  # print("Best Model Coeficients: ", best_model.coef_)
  return best_model

In [ ]:
linear_elastic_reg_model = ElasticNet(max_iter=300000)
lin_param_grid = {'alpha': [0.05, 0.1, 0.2, 0.3, 0.5, 1, 3, 5, 10, 25, 50, 55, 60, 65, 75, 90, 100],
                'l1_ratio': [0.1, 0.3, 0.5, 0.75, 0.9, 0.95, 1]}

linear_best_model = find_and_fit_best_model(linear_elastic_reg_model, lin_param_grid, X_train_scaled, y_train_log)

y_pred_log = linear_best_model.predict(X_test_scaled)
y_pred = np.exp(y_pred_log)

Best Model: ElasticNet(alpha=0.05, l1_ratio=0.1, max_iter=300000)
Best Parameters: {'alpha': 0.05, 'l1_ratio': 0.1}
Best Model Coeficients:  [-0.         -0.01265995 -0.         -0.00014849 -0.         -0.
 -0.01328422 -0.0057829  -0.02227392 -0.0052948  -0.          0.
 -0.          0.02832355  0.01787487  0.01477359  0.01719397  0.00699235
  0.02040997  0.00075986  0.00406842  0.00264843 -0.          0.00148645
  0.0051623   0.00613076  0.          0.          0.00327346  0.00371539
  0.00718765  0.00868186  0.0089532   0.00085539  0.00834277  0.00667948
  0.          0.03454231  0.00167193  0.01784686  0.0001583   0.
  0.01098772  0.03548624  0.          0.01237092  0.01344666  0.02083284
  0.01575845  0.03997885  0.09182551  0.10842421]


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def calculate_and_print_metrics(y_test, y_pred):
  mse = mean_squared_error(y_test, y_pred)
  rmse = np.sqrt(mse)
  mae = mean_absolute_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)

  print("Mean Squared Error:", mse)
  print("Root Mean Squared Error:", rmse)
  print("Mean Absolute Error:", mae)
  print("R-squared:", r2)

  return mse, rmse, mae, r2

In [ ]:
calculate_and_print_metrics(y_test_log, y_pred_log)

Mean Squared Error: 0.017807425177898482
Root Mean Squared Error: 0.13344446477054972
Mean Absolute Error: 0.08959633804396774
R-squared: 0.8951207474910343


(0.017807425177898482,
 np.float64(0.13344446477054972),
 0.08959633804396774,
 0.8951207474910343)

# **Metrics Analysis**

1) Mean Squared Error

MSE = 0.0178.

MSE computes the average of squared differences between target values and predictions. This metric penalizes large errors heavily. A score of 0.0178 indicates extremely low variance in extreme prediction misses. So it indicates a good fit of the model.

2) Root Mean Squared Error

RMSE = 0.1334.

RMSE applies a square root to the MSE.

3) Mean Absolute Error
MAE = 0.0896.

MAE computes the average absolute distance between actual and predicted values. It is robust to outliers. On average, the model's predictions deviate by only 0.0896 units.

4) R^2 = 0.8951.

The model successfully captures and explains 89.5% of the total variance in the target variable. This indicates strong overall predictive performance.

# **Improvement**

Adding Polynomial Features before scaling using Pipeline API of Scikit Learn. I tried the script with degrees 2,3,4, but degrees 3+ was taking impossibly long time to learn with the number of features we have, so I sticked with degree 2, trying both with and without bias.

The results were slightly better than linear.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=True)),
    ('scaler', StandardScaler()),
    ('elasticnet', ElasticNet(max_iter=300000, random_state=13))
])

poly_param_grid = {
    'elasticnet__alpha': [0.001, 0.01, 0.05, 0.1, 1, 10],
    'elasticnet__l1_ratio': [0.05, 0.1, 0.3, 0.5, 0.6, 0.9, 1.0]
}

poly_best_model = find_and_fit_best_model(poly_pipeline, poly_param_grid, X_train_scaled, y_train_log)
y_pred_log = poly_best_model.predict(X_test_scaled)
y_pred = np.exp(y_pred_log)

calculate_and_print_metrics(y_test_log, y_pred_log)

Best Model: Pipeline(steps=[('poly', PolynomialFeatures()), ('scaler', StandardScaler()),
                ('elasticnet',
                 ElasticNet(alpha=0.01, l1_ratio=0.6, max_iter=300000,
                            random_state=13))])
Best Parameters: {'elasticnet__alpha': 0.01, 'elasticnet__l1_ratio': 0.6}
Mean Squared Error: 0.016540948800690743
Root Mean Squared Error: 0.1286116200064782
Mean Absolute Error: 0.0879084059892103
R-squared: 0.9025798323634878


(0.016540948800690743,
 np.float64(0.1286116200064782),
 0.0879084059892103,
 0.9025798323634878)

# Conclusions:

Increasing the polynomial degree reduces bias and slightly improves metrics. However, the applied 60% L1 and 40% L2 regularization blend prevents overfitting by shrinking uninformative polynomial weights to zero, which explains the minimal overall performance gain.